# Compute x*x+y in superposition

Goal: evaluate an integer expression on all 16 input pairs coherently. Declare unsigned two-bit inputs, prepare them uniformly, then initialize a fresh result with `|=`. This is reversible quantum computation: the inputs remain available, and the result width is inferred. It is not a classical loop over 16 executions.

In [ ]:
import flagquantum as fq
from flagquantum import synth
from flagquantum.synth import Const, Output, QNum, allocate, hadamard_transform, qfunc

@qfunc
def energy(x: Const[QNum], y: Const[QNum], result: Output[QNum]) -> None:
    result |= x*x + y

@qfunc
def main(x: Output[QNum], y: Output[QNum], result: Output[QNum]) -> None:
    allocate(2, x)
    allocate(2, y)
    hadamard_transform(x)
    hadamard_transform(y)
    energy(x, y, result)


## Synthesize the arithmetic

The compiler selects arithmetic circuits and temporary storage within the bounds. You specify the expression and quantum data; you do not choose carry wires or write cleanup gates.

In [ ]:
program = synth.synthesize(main, max_qubits=24, memory_limit_bytes=64*1024**2)
result = fq.run(program)
outcomes = dict(program.probabilities(result, registers=("x", "y", "result")))
print(outcomes)
print(program.report["resources"])


## Interpret and modify

There should be 16 outcomes `(x, y, x*x+y)`, each of probability 1/16. The maximum result is 12. `Const` promises the helper preserves its borrowed inputs; it does not turn them into classical values. Replace the expression with `x+y` and inspect how the actual resource report changes. Exact gate counts can change with compiler improvements.